In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}")

In [0]:
from pyspark.sql.functions import lit, current_timestamp
updated_bronze_df=bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp",current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
load_type_norm = load_type.strip().upper()

# Number of updated/new records written to silver (set per load type below)
record_count = 0

if load_type_norm == "FULL":
    # Full load: overwrite the target table
    record_count = updated_bronze_df.count()
    updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type_norm == "APPEND":
    # Append: add incoming records to the target table
    record_count = updated_bronze_df.count()
    updated_bronze_df.write.mode("append").saveAsTable(target_table)

elif load_type_norm == "MERGE":
    # Merge (upsert): update existing records and insert new ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

    missing = [k for k in keys if k not in updated_bronze_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")

    # Deduplicate source on merge keys so each target row matches at most one source row
    source_df = updated_bronze_df.dropDuplicates(keys)

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table (count after dedup = rows actually written)
        record_count = source_df.count()
        source_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp; refresh all other columns
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}

        target_dt = DeltaTable.forName(spark, target_table)
        (target_dt.alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute())

        # Actual rows changed in silver = updated + inserted, from the MERGE commit metrics
        last_op = target_dt.history(1).select("operation", "operationMetrics").collect()[0]
        metrics = last_op["operationMetrics"] or {}
        rows_updated = int(metrics.get("numTargetRowsUpdated", 0))
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        record_count = rows_updated + rows_inserted
        print(f"MERGE metrics -> updated: {rows_updated}, inserted: {rows_inserted}")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

print(f"{load_type_norm} load completed for {target_table}. record_count = {record_count}")

In [0]:
# Return the count of updated/new records to the caller (job / ADF / dbutils.notebook.run)
dbutils.notebook.exit(str(record_count))